# Notebook 02 — Chunking & Metadata

This notebook takes the 1459 parsed BOQ rows from the parser output and converts them into rich, structured chunks ready for embedding and storage in Qdrant.

## What is a "chunk" in our RAG context?

For this BOQ rate intelligence system, a "chunk" is not just text - it is an entire BOQ line item enriched with contextual metadata, normalized fields, and a composite embedding text that combines all relevant information about the item. Each chunk represents one complete bill of quantities line item that can be retrieved during semantic search queries.

Input: `../data/processed/boq_line_items.csv`  
Output: Enriched chunks ready for vector embedding.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import json
import hashlib
import uuid

# Set data path
DATA_PATH = Path("../data/processed/boq_line_items.csv")

# Load CSV
df = pd.read_csv(DATA_PATH)

print(f"Loaded BOQ line items: {df.shape[0]} rows, {df.shape[1]} columns")
print()
print(df.head(3).to_string())

Loaded BOQ line items: 1459 rows, 11 columns

   SNO DESCRIPTION_SHORT SECTION_TITLE                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   SPEC_TEXT                                                                                                                                                                                                                                                                                                                                                                                                        

## Problem 1 — UNIT inconsistency

The parsed BOQ data has inconsistent unit representations. For example:
- Square feet appears as `Sft`, `SFT`, `Sft.`, `sft`
- Numbers appear as `Nos`, `No`, `No's`, `NOS`
- Running feet appears as `rft`, `RFT`, `Rft.`

These need to be normalized to consistent values so that semantic search correctly groups items with the same unit of measurement.

In [2]:
def normalize_unit(unit):
    if pd.isna(unit):
        return None
    
    unit_clean = str(unit).strip().lower()
    
    # Unit mappings
    unit_mappings = {
        # Square feet
        "sft": ["sft", "sft.", "sqft", "sq ft", "sq.ft"],
        # Numbers / Units
        "nos": ["nos", "nos.", "no", "no.", "no's", "unit", "units"],
        # Running feet / Linear feet
        "rft": ["rft", "rft.", "lft", "linear ft"],
        # Lump Sum
        "ls": ["ls", "l.s", "lump sum", "lumpsum"],
        # Cubic feet
        "cft": ["cft", "cu ft", "cu.ft", "cubic ft"],
        # Job / Item
        "job": ["job", "item"],
        # Meters
        "mtr": ["mtr", "mtr.", "meter", "meters"],
        # Bags
        "bags": ["bag", "bags"],
        # Kilograms
        "kgs": ["kg", "kgs", "kilogram", "kilograms"],
        # Points
        "points": ["point", "points"],
        # Each
        "each": ["each", "ea"]
    }
    
    # Check mappings
    for standard, variations in unit_mappings.items():
        if unit_clean in variations:
            return standard
    
    # Fallback: return cleaned lowercase
    return unit_clean

# Apply normalization
print("Before normalization - top 15 units:")
print(df['UNIT'].value_counts().head(15).to_string())
print()

df['UNIT_NORMALIZED'] = df['UNIT'].apply(normalize_unit)

print("After normalization - top 15 units:")
print(df['UNIT_NORMALIZED'].value_counts().head(15).to_string())

Before normalization - top 15 units:
UNIT
Sft      330
SFT      273
Nos      120
Nos.     101
No        87
Rft       73
Job       65
L.S       57
RFT       41
Cft       36
Rft.      28
Nos.      26
1.0       21
Bags      17
Each      12

After normalization - top 15 units:
UNIT_NORMALIZED
sft       612
nos       351
rft       144
job        67
ls         61
cft        36
1.0        21
bags       17
mtr        14
each       12
points     12
lot         6
2.0         5
4.0         5
7.0         5


## Problem 2 — RATE < Rs.10 are invalid

During parsing we noticed many line items with extremely low rates (< ₹10). These are almost certainly component rates or material rates that were incorrectly extracted as full line items.

For rate intelligence purposes, we only want complete work item rates that represent actual billable items.

In [3]:
# Show low rate items
low_rate = df[df['RATE'] < 10]
print(f"Found {len(low_rate)} items with RATE < 10:")
print()
print(low_rate[['DESCRIPTION_SHORT', 'RATE']].head(20).to_string())
print()

# Filter them out
original_count = len(df)
df = df[df['RATE'] >= 10].reset_index(drop=True)
removed_count = original_count - len(df)

print(f"Removed {removed_count} low rate rows")
print(f"Kept {len(df)} valid rows")

Found 30 items with RATE < 10:

                        DESCRIPTION_SHORT      RATE
216           Tools, Equipment & Hardware  5.000000
218                             Form Work  4.950000
219                        Water & Curing  2.000000
220                     Material Shifting  3.000000
221  Lifting, Freight, Cartage & Cleaning  5.000000
227                                  Sand  4.999420
232                                  Sand  3.072563
234           Tools, Equipment & Hardware  5.000000
239                        Water & Curing  2.000000
240                     Material Shifting  3.000000
241  Lifting, Freight, Cartage & Cleaning  5.000000
248                                  Sand  4.519083
252           Tools, Equipment & Hardware  3.000000
255                        Water & Curing  2.000000
256                     Material Shifting  5.000000
264                                  Sand  4.519083
268           Tools, Equipment & Hardware  3.000000
270                           Sc

## Problem 3 — Description quality issues

Many parsed items have quality issues that will affect embedding quality:

1. **`DESCRIPTION_SHORT == DESCRIPTION_FULL`** - No additional detail was extracted
2. **Empty `SPEC_TEXT`** - No specification details available
3. **Very short descriptions** - Insufficient text for good semantic matching

These items still need to be included, but we will need special handling when constructing the embedding text.

In [4]:
# Calculate quality metrics
same_short_full = (df['DESCRIPTION_SHORT'] == df['DESCRIPTION_FULL']).sum()
null_spec = df['SPEC_TEXT'].isna().sum()
short_description = (df['DESCRIPTION_FULL'].str.len() < 30).sum()

print(f"Rows where SHORT == FULL: {same_short_full} ({same_short_full/len(df)*100:.1f}%)")
print(f"Rows with null SPEC_TEXT: {null_spec} ({null_spec/len(df)*100:.1f}%)")
print(f"Rows with DESCRIPTION_FULL < 30 chars: {short_description} ({short_description/len(df)*100:.1f}%)")
print()

print("=== Examples of SHORT == FULL ===")
same_examples = df[df['DESCRIPTION_SHORT'] == df['DESCRIPTION_FULL']].head(3)
print(same_examples[['DESCRIPTION_SHORT', 'DESCRIPTION_FULL']].to_string())
print()

print("=== Examples of null SPEC_TEXT ===")
null_spec_examples = df[df['SPEC_TEXT'].isna()].head(3)
print(null_spec_examples[['DESCRIPTION_FULL', 'SPEC_TEXT']].to_string())
print()

print("=== Examples of short descriptions ===")
short_examples = df[df['DESCRIPTION_FULL'].str.len() < 30].head(3)
print(short_examples[['DESCRIPTION_FULL']].to_string())
print(short_examples['DESCRIPTION_FULL'].str.len().to_string())

Rows where SHORT == FULL: 166 (11.6%)
Rows with null SPEC_TEXT: 920 (64.4%)
Rows with DESCRIPTION_FULL < 30 chars: 17 (1.2%)

=== Examples of SHORT == FULL ===
                                        DESCRIPTION_SHORT                                      DESCRIPTION_FULL
411      Installation of  ST-03 @ Powder room elevation E      Installation of  ST-03 @ Powder room elevation E
412      Installation of  ST-03 @ Powder room elevation W      Installation of  ST-03 @ Powder room elevation W
413  Installation of ST-01 Upto + 1'-6" @ ENT Lobby ELE_N  Installation of ST-01 Upto + 1'-6" @ ENT Lobby ELE_N

=== Examples of null SPEC_TEXT ===
                                                                                                                                                                                                                                                                                                                                                                    

What is a BOQ Chunk for RAG?
Each chunk = one line item. It needs:
1. embedding_text: what gets embedded (rich description)
2. metadata: what gets returned when retrieved
3. chunk_id: unique stable identifier

In [5]:
def build_embedding_text(row):
    """Build embedding text by combining all description signals with context labels."""
    parts = []
    parts.append(f"WORK: {row['WORK_CATEGORY']}")
    parts.append(f"ITEM: {str(row['DESCRIPTION_SHORT']).strip()}")
    if pd.notna(row.get('SPEC_TEXT')) and len(str(row['SPEC_TEXT'])) > 10:
        spec = str(row['SPEC_TEXT'])[:500]  # cap at 500 chars
        parts.append(f"SPEC: {spec}")
    if pd.notna(row.get('SECTION_TITLE')):
        parts.append(f"SECTION: {row['SECTION_TITLE']}")
    parts.append(f"UNIT: {row['UNIT_NORMALIZED']}")
    return "\n".join(parts)

# Apply to all rows
df['embedding_text'] = df.apply(build_embedding_text, axis=1)

# Print examples
print("=== Embedding Text Examples ===")
print("=" * 80)
for i in range(5):
    print(f"Example {i+1}:")
    print(df.iloc[i]['embedding_text'])
    print("-" * 60)

avg_len = df['embedding_text'].str.len().mean()
print(f"\nAverage embedding text length: {avg_len:.0f} characters")

=== Embedding Text Examples ===
Example 1:
WORK: civil_id
ITEM: BRICKWORK 4 1/2"
SPEC: Providing and laying brick masonry using first-class, laboratory-tested bricks having a maximum water absorption of ≤ 20% by weight and a minimum compressive strength of ≥ 2000 psi (≈ 14 MPa), laid in cement–sand mortar in the ratio of 1:4 (Bestway Cement with approved quality sieved sand). Work includes soaking of bricks, proper bonding, raking of joints, maintaining plumb/level/line, curing, scaffolding, and completing the work in all respects as per specifications and instructions of the Engi
SECTION: BRICKWORK
UNIT: sft
------------------------------------------------------------
Example 2:
WORK: civil_id
ITEM: BRICKWORK 9"
SPEC: Providing and laying brick masonry using first-class, laboratory-tested bricks having a maximum water absorption of ≤ 20% by weight and a minimum compressive strength of ≥ 2000 psi (≈ 14 MPa), laid in cement–sand mortar in the ratio of 1:4 (Bestway Cement with approved q

chunk_id must be stable — same 
item from same file always gets same ID. We use MD5 
hash of (source_file + sheet + description_short + rate)

In [6]:
def make_chunk_id(row):
    """Generate stable MD5 hash chunk identifier."""
    key = f"{row['SOURCE_FILE']}|{row['SHEET_NAME']}|{str(row['DESCRIPTION_SHORT'])}|{row['RATE']}"
    return hashlib.md5(key.encode()).hexdigest()

# Apply chunk id
df['chunk_id'] = df.apply(make_chunk_id, axis=1)

# Check duplicates
unique_ids = df['chunk_id'].nunique()
total_rows = len(df)
print(f"Unique chunk_ids: {unique_ids} / Total rows: {total_rows}")

if unique_ids < total_rows:
    duplicates = df[df.duplicated('chunk_id', keep=False)]
    print(f"\nFound {len(duplicates)} duplicate rows:")
    print(duplicates[['DESCRIPTION_SHORT', 'RATE', 'SOURCE_FILE']].to_string())

Unique chunk_ids: 1346 / Total rows: 1429

Found 139 duplicate rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                         DESCRIPTION_SHORT           RATE                SOURCE_FILE
216                                                                                                                                                                                                                                                                                                                                                                                                                           

Now build the complete chunk 
payload — this is exactly what goes into Qdrant as 
payload (metadata) alongside the vector.

In [7]:
# Build final chunks DataFrame
df_chunks = pd.DataFrame()

df_chunks['chunk_id'] = df['chunk_id']
df_chunks['embedding_text'] = df['embedding_text']
df_chunks['description_short'] = df['DESCRIPTION_SHORT']
df_chunks['description_full'] = df['DESCRIPTION_FULL']
df_chunks['section_title'] = df['SECTION_TITLE']
df_chunks['work_category'] = df['WORK_CATEGORY']
df_chunks['rate'] = df['RATE'] 
df_chunks['unit_norm'] = df['UNIT_NORMALIZED']
df_chunks['qty'] = df['QTY']
df_chunks['source_file'] = df['SOURCE_FILE']
df_chunks['sheet_name'] = df['SHEET_NAME']
df_chunks['rate_per_unit_label'] = df.apply(lambda r: f"Rs. {r['RATE']:,.0f} per {r['UNIT_NORMALIZED']}", axis=1)

print(f"Final chunks shape: {df_chunks.shape}")
print("\nData types:")
print(df_chunks.dtypes.to_string())
print("\nFirst 3 chunks:")
print(df_chunks.head(3).to_string())

Final chunks shape: (1429, 12)

Data types:
chunk_id                   str
embedding_text             str
description_short          str
description_full           str
section_title              str
work_category              str
rate                   float64
unit_norm                  str
qty                        str
source_file                str
sheet_name                 str
rate_per_unit_label        str

First 3 chunks:
                           chunk_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

Sanity check — let's see what 
a real search query would match. If someone asks about 
'brick work', which chunks should come up?

In [8]:
def keyword_search(df, query, top_n=5):
    """Simple keyword search test for embedding text."""
    query_lower = query.lower()
    mask = df['embedding_text'].str.lower().str.contains(
        query_lower, na=False)
    results = df[mask].head(top_n)
    
    print(f"\n{'='*60}")
    print(f"QUERY: '{query}'")
    print(f"{'='*60}")
    
    if len(results) == 0:
        print("  No results found")
        return
    
    for _, row in results.iterrows():
        print(f"  ITEM:  {row['description_short']}")
        print(f"  RATE:  {row['rate_per_unit_label']}")
        print(f"  FILE:  {row['source_file']}")
        print()

# Test with common queries
keyword_search(df_chunks, "brick work")
keyword_search(df_chunks, "gypsum ceiling")
keyword_search(df_chunks, "split ac")
keyword_search(df_chunks, "marble")
keyword_search(df_chunks, "wiring")


QUERY: 'brick work'
  ITEM:  9" Brick Works
  RATE:  Rs. 520 per sft
  FILE:  1.Civil.xlsx

  ITEM:  Dismantling of Existing Brick Works of Toilets including shifting of Debries from Site
  RATE:  Rs. 345,000 per ls
  FILE:  BOQ - 01 .xlsx

  ITEM:  Brick Work 4.5" Thick
  RATE:  Rs. 315 per sft
  FILE:  BOQ - 01 .xlsx

  ITEM:  Brick Work 9" Thick
  RATE:  Rs. 485 per sft
  FILE:  BOQ - 01 .xlsx

  ITEM:  Concrete Planters (19'-3" x 5'-6" x 3'-6" High)
  RATE:  Rs. 421,000 per nos
  FILE:  ID Works (G.F).xlsx


QUERY: 'gypsum ceiling'
  No results found

QUERY: 'split ac'
  No results found

QUERY: 'marble'
  ITEM:  Kitchen Shelf Marble Installation
  RATE:  Rs. 220 per sft
  FILE:  1.Civil.xlsx

  ITEM:  Parapet Marble Ledge
  RATE:  Rs. 370 per sft
  FILE:  1.Civil.xlsx

  ITEM:  Servant Passage Marble Installation
  RATE:  Rs. 250 per sft
  FILE:  1.Civil.xlsx

  ITEM:  THRESHOLD (Base Rate Sft @ 700)
  RATE:  Rs. 1,350 per sft
  FILE:  2. FLOORING.xlsx

  ITEM:  STAIR(s)  (Base R

Rate distribution analysis — 
understanding our knowledge base by work category

In [9]:
# Rate statistics per work category
print("=" * 80)
print("RATE STATISTICS BY WORK CATEGORY")
print("=" * 80)

for category, group in df_chunks.groupby('work_category'):
    count = len(group)
    min_rate = group['rate'].min()
    max_rate = group['rate'].max()
    median_rate = group['rate'].median()
    sample = group.iloc[0]
    
    print(f"\n{category.upper()}:")
    print(f"  Count:    {count}")
    print(f"  Min:      Rs. {min_rate:,.0f}")
    print(f"  Max:      Rs. {max_rate:,.0f}")
    print(f"  Median:   Rs. {median_rate:,.0f}")
    print(f"  Sample:   {sample['description_short']} @ Rs. {sample['rate']:,.0f}")
    print("-" * 60)

print(f"\n{'='*80}")
print(f"Knowledge base snapshot — static")
print(f"Total chunks: {len(df_chunks)}")
print(f"{'='*80}")

RATE STATISTICS BY WORK CATEGORY

CIVIL_ID:
  Count:    648
  Min:      Rs. 25
  Max:      Rs. 1,424,500
  Median:   Rs. 625
  Sample:   BRICKWORK 4 1/2" @ Rs. 310
------------------------------------------------------------

ELECTRICAL_ELV:
  Count:    246
  Min:      Rs. 155
  Max:      Rs. 2,185,000
  Median:   Rs. 11,750
  Sample:   Wiring of camera with CAT-6 UTP cable in 1" dia PVC pipe/on cable tray or as per site requirements from RJ45 outlet (for camera) to patch panel including cost of RJ 45 outlet with 16 SWG sheet steel back box recessed in wall / wall surface, complete in all respects. @ Rs. 16,500
------------------------------------------------------------

FIRE_FIGHTING:
  Count:    19
  Min:      Rs. 770
  Max:      Rs. 525,000
  Median:   Rs. 7,350
  Sample:   Supply at site, installation, testing and commissioning of Single loop Intelligent Addressable Fire Alarm Control Panel (FACP) can be expandable, complete with built-in power supply unit, NICAD battery backup, A

Embedding text length analysis — 
important for choosing embedding model token limits. 
text-embedding-3-large supports 8191 tokens.

In [10]:
# Token length analysis
df_chunks['est_tokens'] = df_chunks['embedding_text'].str.len() / 4

print("=" * 80)
print("TOKEN LENGTH ANALYSIS")
print("=" * 80)

print(f"Min tokens:  {df_chunks['est_tokens'].min():.0f}")
print(f"Max tokens:  {df_chunks['est_tokens'].max():.0f}")
print(f"Mean tokens: {df_chunks['est_tokens'].mean():.0f}")

# Check token thresholds
over_512 = len(df_chunks[df_chunks['est_tokens'] > 512])
over_1024 = len(df_chunks[df_chunks['est_tokens'] > 1024])
over_8191 = len(df_chunks[df_chunks['est_tokens'] > 8191])

print(f"\nRows > 512 tokens:  {over_512}")
if over_512 > 0:
    print("Examples:")
    for _, row in df_chunks[df_chunks['est_tokens'] > 512].head(2).iterrows():
        print(f"  {row['description_short']}: {row['est_tokens']:.0f} tokens")

print(f"Rows > 1024 tokens: {over_1024}")
print(f"Rows > 8191 tokens: {over_8191} (should be 0)")

# Text histogram
print("\n" + "=" * 80)
print("TOKEN LENGTH DISTRIBUTION")
print("=" * 80)

buckets = [
    ("0-100", 0, 100),
    ("100-300", 100, 300),
    ("300-512", 300, 512),
    ("512-1024", 512, 1024),
    ("1024+", 1024, float('inf'))
]

for label, low, high in buckets:
    count = len(df_chunks[(df_chunks['est_tokens'] >= low) & (df_chunks['est_tokens'] < high)])
    pct = count / len(df_chunks) * 100
    bar = '█' * int(pct / 5)
    print(f"{label:>8} | {bar:<20} | {count:4d} ({pct:4.1f}%)")

TOKEN LENGTH ANALYSIS
Min tokens:  15
Max tokens:  812
Mean tokens: 94

Rows > 512 tokens:  1
Examples:
  Supply, installation and commissioning of MPB  made of sheet   steel   14   SWG,   totally   enclosed,   indoor   Floor Standing  type   including  all  auxiliaries,  internal  wiring from   MCCBs   terminating   on   cable   terminals   blocks installed  at  the  top  of  MPB  designation  labels  of  all incoming   and   outgoing   feeders   as   shown   on   the drawings,  Brass  Cable  Glands  according  to  the  Cable sizes  for   incoming  &   outgoing  Cables,   earthing   bar, neutral bar, suitable for system voltage 415 V, 50 Hz, 3 phase,  neutral  and  earthing  busbars  made  of  99.8% electrolytic copper of under mentioned capacities. Body of  MPB  shall  be  degreased  and  de‐rusted  having  one coat  of  antirust  paint  with  further  2  coats  of  powder paint  of   approved   colour   and  shall  be   equipped   as mentioned    below,    including   cost   of   al

In [11]:
# Save final chunks
OUTPUT_CSV = Path("../data/processed/boq_chunks.csv")
OUTPUT_JSON = Path("../data/processed/boq_chunks.json")

# Ensure directory exists
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Save CSV
df_chunks.to_csv(OUTPUT_CSV, index=False)

# Save JSON
df_chunks.to_json(OUTPUT_JSON, orient='records', indent=2)

# Print confirmation
print("✓ CSV saved: data/processed/boq_chunks.csv")
print("✓ JSON saved: data/processed/boq_chunks.json")
print(f"  Total chunks: {len(df_chunks)}")
print(f"  Columns: {df_chunks.columns.tolist()}")
print("  Ready for Notebook 03 — Qdrant Embeddings")

✓ CSV saved: data/processed/boq_chunks.csv
✓ JSON saved: data/processed/boq_chunks.json
  Total chunks: 1429
  Columns: ['chunk_id', 'embedding_text', 'description_short', 'description_full', 'section_title', 'work_category', 'rate', 'unit_norm', 'qty', 'source_file', 'sheet_name', 'rate_per_unit_label', 'est_tokens']
  Ready for Notebook 03 — Qdrant Embeddings


## What We Built
- Total chunks ready for embedding: N
- Work categories covered: list them
- embedding_text format explained
- chunk_id strategy explained
- Next: Notebook 03 will embed these into Qdrant
  using text-embedding-3-large and build hybrid search